# Notebook 9 - Real-Time Interactive Dashboard (Improved)
**Project**: Real-Time Retail Analytics and Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

**Improvements**: Better heatmap colors, country filter, interactive charts, new layout

Dashboard URL: **http://localhost:8050**


In [1]:
import sys
!{sys.executable} -m pip install dash==2.17.0 dash-bootstrap-components==1.6.0 plotly==5.20.0 deltalake pyarrow boto3 joblib -q
print('Done.')

Done.


In [2]:
import pandas as pd
import numpy as np
import time
import threading
import warnings
warnings.filterwarnings('ignore')
from deltalake import DeltaTable

STORAGE = {
    'AWS_ENDPOINT_URL':           'http://minio:9000',
    'AWS_ACCESS_KEY_ID':          'admin',
    'AWS_SECRET_ACCESS_KEY':      'bigdata123',
    'AWS_ALLOW_HTTP':             'true',
    'AWS_S3_ALLOW_UNSAFE_RENAME': 'true',
    'AWS_REGION':                 'us-east-1'
}

print('Loading data from Delta Lake...')
df_tx = DeltaTable('s3://retail-v2/delta/transactions', storage_options=STORAGE).to_pandas()
df_tx['InvoiceDate'] = pd.to_datetime(df_tx['InvoiceDate'])
df_tx['Revenue'] = pd.to_numeric(df_tx['Revenue'], errors='coerce')
df_tx['Quantity'] = pd.to_numeric(df_tx['Quantity'], errors='coerce')
print(f'Loaded {len(df_tx):,} rows from Delta Lake')

df_tx['Date']      = df_tx['InvoiceDate'].dt.date.astype(str)
df_tx['MonthStr']  = df_tx['InvoiceDate'].dt.strftime('%Y-%m')
df_tx['Hour']      = pd.to_numeric(df_tx['Hour'], errors='coerce').fillna(0).astype(int)
df_tx['DayOfWeek'] = pd.to_numeric(df_tx['DayOfWeek'], errors='coerce').fillna(0).astype(int)
if 'DayName' not in df_tx.columns:
    df_tx['DayName'] = df_tx['InvoiceDate'].dt.day_name()

# Pre-compute aggregations
daily = df_tx.groupby('Date').agg(Revenue=('Revenue','sum'), Tx=('InvoiceNo','nunique')).reset_index().sort_values('Date')
daily['Date'] = pd.to_datetime(daily['Date'])
daily['MA7'] = daily['Revenue'].rolling(7, min_periods=1).mean()

country = df_tx.groupby('Country').agg(Revenue=('Revenue','sum'), Tx=('InvoiceNo','count'), Cust=('CustomerID','nunique')).reset_index().sort_values('Revenue', ascending=False).head(15)
top_prod = df_tx.groupby(['StockCode','Description']).agg(Qty=('Quantity','sum'), Rev=('Revenue','sum')).reset_index().sort_values('Rev', ascending=False).head(15)
heatmap_data = df_tx.groupby(['DayName','Hour'])['Revenue'].sum().reset_index()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
monthly = df_tx.groupby('MonthStr').agg(Rev=('Revenue','sum'), Qty=('Quantity','sum')).reset_index().sort_values('MonthStr')

# Country list for dropdown
all_countries = ['All Countries'] + sorted(df_tx['Country'].unique().tolist())

K = {
    'rev': df_tx['Revenue'].sum(),
    'tx':  df_tx['InvoiceNo'].nunique(),
    'cust': df_tx['CustomerID'].nunique(),
    'aov': df_tx['Revenue'].sum() / max(df_tx['InvoiceNo'].nunique(), 1),
    'top': country.iloc[0]['Country'] if len(country) else 'N/A',
    'products': df_tx['StockCode'].nunique(),
    'countries': df_tx['Country'].nunique()
}
print(f"Revenue: {K['rev']:,.0f} | Tx: {K['tx']:,} | Cust: {K['cust']:,}")
print('Aggregations ready.')

Loading data from Delta Lake...
Loaded 500,000 rows from Delta Lake
Revenue: 10,881,575 | Tx: 22,333 | Cust: 4,649
Aggregations ready.


In [3]:
import joblib
import os

MODEL_PATH = '/home/jovyan/work/model_bundle.joblib'
if os.path.exists(MODEL_PATH):
    bundle = joblib.load(MODEL_PATH)
    ml_model = bundle['model']
    ml_features = bundle['features']
    ml_name = bundle['model_name']
    HAS_MODEL = True
    print(f'ML model loaded: {ml_name}')
else:
    HAS_MODEL = False
    print('No model found.')

ML model loaded: Linear Regression


In [4]:
_stream = {'sent': 0, 'rate': 0, 'start': time.time(), 'lock': threading.Lock()}
def _sim():
    while True:
        with _stream['lock']:
            _stream['sent'] += np.random.randint(200, 500)
            _stream['rate'] = _stream['sent'] / max(time.time() - _stream['start'], 1)
        time.sleep(1)
threading.Thread(target=_sim, daemon=True).start()
print('Stream simulator running.')

Stream simulator running.


In [5]:
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ============================================================
# COLOR THEME
# ============================================================
C = {
    'bg':     '#080b12',
    'card':   '#111827',
    'border': '#1f2b47',
    'text':   '#e2e8f8',
    'sub':    '#6b7fa3',
    'accent': '#3b82f6',
    'green':  '#10b981',
    'yellow': '#f59e0b',
    'red':    '#ef4444',
    'purple': '#8b5cf6',
    'plotbg': '#0c1221',
    'cyan':   '#06b6d4',
    'orange': '#f97316',
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def card(children, p='14px'):
    return html.Div(children, style={
        'backgroundColor': C['card'],
        'border': f"1px solid {C['border']}",
        'borderRadius': '12px',
        'padding': p,
        'height': '100%'
    })

def kpi(icon, label, value, sub, color):
    return card(html.Div([
        html.Div(icon, style={'fontSize': '28px', 'marginBottom': '6px'}),
        html.Div(label, style={'color': C['sub'], 'fontSize': '10px', 'letterSpacing': '1.5px', 'textTransform': 'uppercase'}),
        html.Div(value, style={'color': color, 'fontSize': '22px', 'fontWeight': '800', 'lineHeight': '1.3'}),
        html.Div(sub, style={'color': C['sub'], 'fontSize': '10px', 'marginTop': '2px'}),
    ]))

def chart_title(title, subtitle=None):
    children = [html.Div(title, style={'color': C['text'], 'fontWeight': '700', 'fontSize': '13px'})]
    if subtitle:
        children.append(html.Div(subtitle, style={'color': C['sub'], 'fontSize': '10px', 'marginTop': '2px'}))
    return html.Div(children, style={'marginBottom': '8px'})

inp_s = {'width': '100%', 'backgroundColor': C['bg'], 'color': C['text'],
         'border': f"1px solid {C['border']}", 'borderRadius': '6px', 'padding': '6px', 'fontSize': '12px'}

# ============================================================
# APP
# ============================================================
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY], title='Retail Analytics | Z5008')

app.layout = html.Div([
    dcc.Interval(id='tick', interval=3000, n_intervals=0),

    # ── HEADER ──────────────────────────────────────────────
    html.Div([
        html.Div([
            html.H3('Real-Time Retail Analytics', style={'color': C['text'], 'margin': '0', 'fontWeight': '800', 'fontSize': '20px'}),
            html.Small('IIT Madras Zanzibar | Z5008 | Vineet Joshi | ZDA25M007', style={'color': C['sub']}),
        ]),
        html.Div([
            html.Span(id='live-dot', style={'color': C['green'], 'fontWeight': '700', 'marginRight': '14px', 'fontSize': '13px'}),
            html.Span(id='hdr-time', style={'color': C['sub'], 'fontSize': '11px'}),
        ])
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center',
               'backgroundColor': C['card'], 'borderBottom': f"1px solid {C['border']}",
               'padding': '14px 24px', 'marginBottom': '12px'}),

    html.Div([

        # ── FILTER BAR ──────────────────────────────────────
        dbc.Row([
            dbc.Col(card(html.Div([
                html.Div('FILTER BY COUNTRY', style={'color': C['accent'], 'fontSize': '10px', 'fontWeight': '700', 'letterSpacing': '1.5px', 'marginBottom': '6px'}),
                dcc.Dropdown(
                    id='country-filter',
                    options=[{'label': c, 'value': c} for c in all_countries],
                    value='All Countries',
                    clearable=False,
                    style={'backgroundColor': C['bg'], 'color': C['text']}
                )
            ]), p='10px'), md=4),
            dbc.Col(card(html.Div([
                html.Div('FILTER BY YEAR', style={'color': C['accent'], 'fontSize': '10px', 'fontWeight': '700', 'letterSpacing': '1.5px', 'marginBottom': '6px'}),
                dcc.Dropdown(
                    id='year-filter',
                    options=[{'label': 'All Years', 'value': 'All'}] + [{'label': str(y), 'value': y} for y in sorted(df_tx['Year'].unique())],
                    value='All',
                    clearable=False,
                    style={'backgroundColor': C['bg'], 'color': C['text']}
                )
            ]), p='10px'), md=4),
            dbc.Col(card(html.Div([
                html.Div('CHART TYPE - HEATMAP', style={'color': C['accent'], 'fontSize': '10px', 'fontWeight': '700', 'letterSpacing': '1.5px', 'marginBottom': '6px'}),
                dcc.RadioItems(
                    id='heatmap-metric',
                    options=[
                        {'label': ' Revenue', 'value': 'Revenue'},
                        {'label': ' Quantity', 'value': 'Quantity'},
                        {'label': ' Transactions', 'value': 'Tx'},
                    ],
                    value='Revenue',
                    inline=True,
                    style={'color': C['text'], 'fontSize': '12px'}
                )
            ]), p='10px'), md=4),
        ], className='mb-3 g-2'),

        # ── KPI CARDS ───────────────────────────────────────
        dbc.Row([
            dbc.Col(kpi('\U0001F4B7', 'Revenue', f"\u00A3{K['rev']/1e6:.2f}M", 'All time', C['green']), md=2),
            dbc.Col(kpi('\U0001F9FE', 'Transactions', f"{K['tx']:,}", 'Unique invoices', C['accent']), md=2),
            dbc.Col(kpi('\U0001F465', 'Customers', f"{K['cust']:,}", 'Unique IDs', C['yellow']), md=2),
            dbc.Col(kpi('\U0001F3C6', 'AOV', f"\u00A3{K['aov']:.2f}", f"Top: {K['top']}", C['purple']), md=2),
            dbc.Col(kpi('\U0001F4E6', 'Products', f"{K['products']:,}", 'Unique SKUs', C['cyan']), md=2),
            dbc.Col(kpi('\U0001F30D', 'Countries', f"{K['countries']}", 'Markets served', C['red']), md=2),
        ], className='mb-3 g-2'),

        # ── ROW 2: Revenue Trend + Country Bar ──────────────
        dbc.Row([
            dbc.Col(card([
                chart_title('Daily Revenue Trend', 'Click a point to see details'),
                dcc.Graph(id='g-trend', config={'displayModeBar': True, 'modeBarButtonsToRemove': ['pan2d','lasso2d']}, style={'height': '260px'})
            ]), md=8),
            dbc.Col(card([
                chart_title('Revenue by Country', 'Filtered by selection'),
                dcc.Graph(id='g-country', config={'displayModeBar': False}, style={'height': '260px'})
            ]), md=4),
        ], className='mb-3 g-2'),

        # ── ROW 3: Heatmap + Top Products ───────────────────
        dbc.Row([
            dbc.Col(card([
                chart_title('Sales Heatmap — Day x Hour', 'Toggle metric above | Hover for details'),
                dcc.Graph(id='g-heat', config={'displayModeBar': False}, style={'height': '280px'})
            ]), md=6),
            dbc.Col(card([
                chart_title('Top 10 Products by Revenue', 'Hover to see full name'),
                dcc.Graph(id='g-prod', config={'displayModeBar': False}, style={'height': '280px'})
            ]), md=6),
        ], className='mb-3 g-2'),

        # ── ROW 4: Monthly + Predictor ───────────────────────
        dbc.Row([
            dbc.Col(card([
                chart_title('Monthly Revenue & Volume', 'Dual axis — bars=revenue, line=quantity'),
                dcc.Graph(id='g-month', config={'displayModeBar': False}, style={'height': '230px'})
            ]), md=7),
            dbc.Col(card([
                html.Div('\U0001F916 Demand Predictor', style={'color': C['text'], 'fontWeight': '700', 'fontSize': '13px', 'marginBottom': '10px'}),
                html.Div([
                    html.Label('Month', style={'color': C['sub'], 'fontSize': '10px'}),
                    dcc.Dropdown(id='p-m', options=[{'label': m, 'value': i} for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], 1)],
                                 value=11, clearable=False, style={'backgroundColor': C['bg'], 'color': C['text'], 'marginBottom': '6px'}),
                    html.Label('Day of Week', style={'color': C['sub'], 'fontSize': '10px'}),
                    dcc.Dropdown(id='p-d', options=[{'label': d, 'value': i} for i, d in enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], 1)],
                                 value=3, clearable=False, style={'backgroundColor': C['bg'], 'color': C['text'], 'marginBottom': '6px'}),
                    html.Div([
                        html.Div([
                            html.Label('Avg Price (\u00A3)', style={'color': C['sub'], 'fontSize': '10px'}),
                            dcc.Input(id='p-price', type='number', value=3.75, step=0.25, min=0.1, style=inp_s),
                        ], style={'flex': '1', 'marginRight': '8px'}),
                        html.Div([
                            html.Label('Invoices', style={'color': C['sub'], 'fontSize': '10px'}),
                            dcc.Input(id='p-inv', type='number', value=5, step=1, min=1, style=inp_s),
                        ], style={'flex': '1'}),
                    ], style={'display': 'flex', 'marginBottom': '6px'}),
                    html.Button('\U0001F4CA Predict Demand', id='p-btn', n_clicks=0,
                                style={'width': '100%', 'backgroundColor': C['accent'], 'color': 'white',
                                       'border': 'none', 'borderRadius': '8px', 'padding': '10px',
                                       'fontWeight': '700', 'cursor': 'pointer', 'marginTop': '4px', 'fontSize': '13px'}),
                    html.Div(id='p-out', style={'backgroundColor': C['bg'], 'borderRadius': '8px', 'padding': '10px',
                                                'border': f"1px solid {C['border']}", 'textAlign': 'center',
                                                'marginTop': '8px', 'minHeight': '48px'})
                ])
            ]), md=5),
        ], className='mb-3 g-2'),

        # ── ROW 5: Kafka Monitor ─────────────────────────────
        dbc.Row([dbc.Col(card([
            html.Div('\U0001F4E1 Kafka Stream Monitor | kafka:9092 (KRaft) | Topic: retail-txn-v2',
                     style={'color': C['text'], 'fontWeight': '600', 'marginBottom': '10px', 'fontSize': '12px'}),
            html.Div(id='stream-stats')
        ]))], className='mb-3'),

    ], style={'padding': '0 16px'})
], style={'backgroundColor': C['bg'], 'minHeight': '100vh', 'fontFamily': "'Inter','Segoe UI',sans-serif"})

print('Layout built.')

Layout built.


In [6]:
@app.callback(
    Output('g-trend',    'figure'),
    Output('g-country',  'figure'),
    Output('g-heat',     'figure'),
    Output('g-prod',     'figure'),
    Output('g-month',    'figure'),
    Output('hdr-time',   'children'),
    Output('live-dot',   'children'),
    Output('stream-stats','children'),
    Input('tick',          'n_intervals'),
    Input('country-filter','value'),
    Input('year-filter',   'value'),
    Input('heatmap-metric','value'),
)
def refresh(n, sel_country, sel_year, heat_metric):

    # ── Filter data ─────────────────────────────────────────
    dff = df_tx.copy()
    if sel_country and sel_country != 'All Countries':
        dff = dff[dff['Country'] == sel_country]
    if sel_year and sel_year != 'All':
        dff = dff[dff['Year'] == sel_year]

    # ── Recompute on filtered data ───────────────────────────
    f_daily = dff.groupby('Date').agg(Revenue=('Revenue','sum')).reset_index()
    f_daily['Date'] = pd.to_datetime(f_daily['Date'])
    f_daily = f_daily.sort_values('Date')
    f_daily['MA7'] = f_daily['Revenue'].rolling(7, min_periods=1).mean()

    f_country = dff.groupby('Country')['Revenue'].sum().reset_index().sort_values('Revenue', ascending=False).head(15)
    f_top = dff.groupby(['StockCode','Description'])['Revenue'].sum().reset_index().sort_values('Revenue', ascending=False).head(10)
    f_monthly = dff.groupby('MonthStr').agg(Rev=('Revenue','sum'), Qty=('Quantity','sum')).reset_index().sort_values('MonthStr')

    # Heatmap metric
    if heat_metric == 'Quantity':
        f_heat = dff.groupby(['DayName','Hour'])['Quantity'].sum().reset_index()
        f_heat.columns = ['DayName','Hour','Value']
        heat_label = 'Quantity'
    elif heat_metric == 'Tx':
        f_heat = dff.groupby(['DayName','Hour'])['InvoiceNo'].count().reset_index()
        f_heat.columns = ['DayName','Hour','Value']
        heat_label = 'Transactions'
    else:
        f_heat = dff.groupby(['DayName','Hour'])['Revenue'].sum().reset_index()
        f_heat.columns = ['DayName','Hour','Value']
        heat_label = 'Revenue (\u00A3)'

    # ── CHART 1: Revenue Trend ───────────────────────────────
    ft = go.Figure()
    ft.add_trace(go.Scatter(
        x=f_daily['Date'], y=f_daily['Revenue'],
        mode='lines', name='Daily Revenue',
        line=dict(color=C['accent'], width=1.5),
        fill='tozeroy', fillcolor='rgba(59,130,246,0.10)',
        hovertemplate='<b>%{x|%d %b %Y}</b><br>Revenue: \u00A3%{y:,.0f}<extra></extra>'
    ))
    ft.add_trace(go.Scatter(
        x=f_daily['Date'], y=f_daily['MA7'],
        mode='lines', name='7-Day MA',
        line=dict(color=C['yellow'], width=2, dash='dot'),
        hovertemplate='7-Day MA: \u00A3%{y:,.0f}<extra></extra>'
    ))
    ft.update_layout(
        paper_bgcolor=C['plotbg'], plot_bgcolor=C['plotbg'],
        font=dict(color=C['text'], size=11),
        margin=dict(l=50, r=15, t=10, b=35),
        showlegend=True,
        legend=dict(orientation='h', y=1.15, bgcolor='rgba(0,0,0,0)', font_size=10),
        hovermode='x unified'
    )
    ft.update_xaxes(gridcolor=C['border'], showspikes=True, spikecolor=C['accent'], spikethickness=1)
    ft.update_yaxes(gridcolor=C['border'], tickprefix='\u00A3')

    # ── CHART 2: Country Bar ─────────────────────────────────
    colors = px.colors.sequential.Teal
    fc = go.Figure(go.Bar(
        x=f_country['Revenue'], y=f_country['Country'],
        orientation='h',
        marker=dict(
            color=f_country['Revenue'],
            colorscale='Teal',
            showscale=False,
            line=dict(width=0)
        ),
        hovertemplate='<b>%{y}</b><br>Revenue: \u00A3%{x:,.0f}<extra></extra>'
    ))
    fc.update_layout(
        paper_bgcolor=C['plotbg'], plot_bgcolor=C['plotbg'],
        font=dict(color=C['text'], size=11),
        margin=dict(l=10, r=15, t=10, b=35)
    )
    fc.update_xaxes(gridcolor=C['border'], tickprefix='\u00A3')
    fc.update_yaxes(gridcolor=C['border'], categoryorder='total ascending')

    # ── CHART 3: Heatmap (IMPROVED) ─────────────────────────
    pivot = f_heat.pivot_table(
        index='DayName', columns='Hour', values='Value', aggfunc='sum'
    ).reindex(day_order).fillna(0)

    # Use a much better colorscale: Viridis for Revenue, Plasma for Qty, Blues for Tx
    cscale_map = {'Revenue': 'Viridis', 'Quantity': 'Plasma', 'Tx': 'Blues'}
    cscale = cscale_map.get(heat_metric, 'Viridis')

    fh = go.Figure(go.Heatmap(
        z=pivot.values,
        x=[f'{h:02d}:00' for h in range(24)],
        y=day_order,
        colorscale=cscale,
        showscale=True,
        hoverongaps=False,
        hovertemplate='<b>%{y} %{x}</b><br>' + heat_label + ': %{z:,.0f}<extra></extra>',
        colorbar=dict(
            thickness=12,
            len=0.8,
            title=dict(text=heat_label, side='right', font=dict(size=10, color=C['sub'])),
            tickfont=dict(color=C['sub'], size=9)
        )
    ))
    fh.update_layout(
        paper_bgcolor=C['plotbg'], plot_bgcolor=C['plotbg'],
        font=dict(color=C['text'], size=11),
        margin=dict(l=80, r=60, t=10, b=40),
        xaxis=dict(title='Hour of Day', tickangle=-45, gridcolor=C['border']),
        yaxis=dict(title='', gridcolor=C['border'])
    )

    # ── CHART 4: Top Products ────────────────────────────────
    fp = go.Figure(go.Bar(
        x=f_top['Revenue'],
        y=[str(d)[:28] for d in f_top['Description']],
        orientation='h',
        marker=dict(
            color=f_top['Revenue'],
            colorscale='Tealgrn',
            showscale=False,
            line=dict(width=0)
        ),
        text=[f'\u00A3{v/1000:.1f}K' for v in f_top['Revenue']],
        textposition='outside',
        textfont=dict(color=C['sub'], size=9),
        hovertemplate='<b>%{y}</b><br>Revenue: \u00A3%{x:,.0f}<extra></extra>',
        customdata=f_top['Description']
    ))
    fp.update_layout(
        paper_bgcolor=C['plotbg'], plot_bgcolor=C['plotbg'],
        font=dict(color=C['text'], size=10),
        margin=dict(l=10, r=60, t=10, b=35)
    )
    fp.update_xaxes(gridcolor=C['border'], tickprefix='\u00A3')
    fp.update_yaxes(gridcolor=C['border'], categoryorder='total ascending')

    # ── CHART 5: Monthly Dual Axis ───────────────────────────
    fm = make_subplots(specs=[[{'secondary_y': True}]])
    fm.add_trace(go.Bar(
        x=f_monthly['MonthStr'], y=f_monthly['Rev'],
        name='Revenue', marker_color=C['accent'], opacity=0.75,
        hovertemplate='%{x}<br>Revenue: \u00A3%{y:,.0f}<extra></extra>'
    ), secondary_y=False)
    fm.add_trace(go.Scatter(
        x=f_monthly['MonthStr'], y=f_monthly['Qty'],
        name='Quantity', line=dict(color=C['orange'], width=2.5),
        mode='lines+markers', marker=dict(size=5),
        hovertemplate='%{x}<br>Quantity: %{y:,}<extra></extra>'
    ), secondary_y=True)
    fm.update_layout(
        paper_bgcolor=C['plotbg'], plot_bgcolor=C['plotbg'],
        font=dict(color=C['text'], size=11),
        margin=dict(l=50, r=50, t=10, b=35),
        legend=dict(orientation='h', y=1.15, bgcolor='rgba(0,0,0,0)', font_size=10),
        hovermode='x unified'
    )
    fm.update_yaxes(tickprefix='\u00A3', secondary_y=False, gridcolor=C['border'])
    fm.update_yaxes(secondary_y=True, gridcolor=C['border'])
    fm.update_xaxes(gridcolor=C['border'], tickangle=-45)

    # ── STREAM STATS ─────────────────────────────────────────
    with _stream['lock']:
        sent = _stream['sent']
        rate = _stream['rate']

    def sb(label, val, color):
        return html.Div([
            html.Div(label, style={'color': C['sub'], 'fontSize': '9px', 'marginBottom': '3px', 'letterSpacing': '1px'}),
            html.Div(val, style={'color': color, 'fontSize': '16px', 'fontWeight': '800'})
        ], style={'flex': '1', 'padding': '6px 14px', 'borderRight': f"1px solid {C['border']}"})

    stats = html.Div([
        sb('MESSAGES',   f'{sent:,}',            C['green']),
        sb('THROUGHPUT', f'{rate:.0f} rec/s',    C['yellow']),
        sb('BROKER',     'kafka:9092',            C['accent']),
        sb('MODE',       'KRaft',                 C['purple']),
        sb('PARTITIONS', '3',                     C['text']),
        sb('DELTA LAKE', f'{len(df_tx):,} rows', C['cyan']),
        sb('FILTERED',   f'{len(dff):,} rows',   C['orange']),
    ], style={'display': 'flex', 'flexWrap': 'wrap'})

    ts    = f"Updated {datetime.now().strftime('%H:%M:%S')}"
    blink = '\u2B24 LIVE' if n % 2 == 0 else '\u25CB LIVE'
    return ft, fc, fh, fp, fm, ts, blink, stats


@app.callback(
    Output('p-out', 'children'),
    Input('p-btn', 'n_clicks'),
    State('p-m', 'value'), State('p-d', 'value'),
    State('p-price', 'value'), State('p-inv', 'value'),
    prevent_initial_call=True
)
def predict(n, month, dow, price, inv):
    price = price or 3.75
    inv   = inv or 5
    if HAS_MODEL:
        inp = pd.DataFrame([{
            'Year': 2010, 'Month': month,
            'WeekOfYear': month * 4, 'DayOfWeek': dow,
            'AvgUnitPrice': price, 'NumInvoices': inv,
            'UniqueCustomers': max(1, inv - 1),
            'TotalRevenue': price * inv * 10,
            'StockCode_idx': 50, 'Country_idx': 0
        }])
        qty = max(0, ml_model.predict(inp[ml_features])[0])
        src = ml_name
        col = C['green']
    else:
        seasonal = 1.3 if month in [11, 12] else (0.85 if month in [6, 7] else 1.0)
        qty = max(1, int(inv * (15.0 / max(price, 0.5)) * seasonal))
        src = 'Heuristic'
        col = C['yellow']
    month_names = ['','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    tag = '\U0001F525 Peak Season' if month in [11, 12] else ('\u2744\ufe0f Slow Season' if month in [6, 7] else '\u2705 Normal')
    return html.Div([
        html.Div(f'{qty:,.0f} units', style={'color': col, 'fontSize': '26px', 'fontWeight': '800'}),
        html.Div(f'{tag} | {month_names[month]} | {src}',
                 style={'color': C['sub'], 'fontSize': '11px', 'marginTop': '4px'})
    ])

print('Callbacks registered.')

Callbacks registered.


In [7]:
print('Starting dashboard on http://localhost:8050')
app.run(debug=False, port=8050, host='0.0.0.0')

Starting dashboard on http://localhost:8050
